# Centauri Bloom — post-training step 2: response-level RL (GRPO or best-of-n)

Starts from the **SFT candidate** (the newest `Bloom-Chat-0.2` PR trained on Instruct mix v2, or merged `main`, or a local SFT run folder) and improves whole answers with **rule-based rewards**: checkable correctness on generated tasks (arithmetic, word problems, successor, even/odd, compare, letters, reversing, sorting, unit conversion, capitals, optional GSM8K train), plus format, stopping, repetition and length terms, plus an optional small LLM judge for open-ended prompts. See `docs/post-training.md` for the design.

- `grpo`: sample a group of answers per prompt, reward each whole answer, push up above-average answers and push down below-average ones, with a KL leash to the SFT model.
- `best-of-n`: keep only the single best answer per prompt (if it clears a bar) and fine-tune on those. Slower to improve, harder to break.

The notebook evaluates on a **locked eval and fixed held-out tasks that training never sees**, keeps the **best** checkpoint, and uploads **only if it beats the parent by a margin** — as a **new PR only**. RL prompts are generated from templates (and public GSM8K train); **Jaron's data is not used or needed here.**

**Kaggle:** GPU T4, Internet on, `HF_TOKEN` secret. Private visibility recommended. About 20–40 minutes on a T4 for 300 steps.

In [ ]:
#@title 1. Settings
ALGO = "grpo" #@param ["grpo","best-of-n"]
STEPS = 300 #@param {type:"integer"}
PROMPTS_PER_STEP = 16 #@param {type:"integer"}
GROUP_SIZE = 8 #@param {type:"integer"}
MAX_NEW = 96 #@param {type:"integer"}
TEMPERATURE = 1.0 #@param {type:"number"}
#@markdown Raise to 1.1–1.3 if the log warns that few groups carry signal.
LEARNING_RATE = 1e-5 #@param {type:"number"}
KL_BETA = 0.02 #@param {type:"number"}
EVAL_EVERY = 25 #@param {type:"integer"}
USE_GSM8K = True #@param {type:"boolean"}
#@markdown Mix in GSM8K *train* questions (MIT). Test split is never used.
JUDGE_MODEL = "" #@param {type:"string"}
#@markdown Optional open-weights judge for open-ended prompts, e.g. `Qwen/Qwen2.5-0.5B-Instruct`. Blank = rules only.
PARENT_HF_REVISION = "auto" #@param {type:"string"}
#@markdown `auto` = newest Bloom-Chat-0.2 PR trained on Instruct mix v2 (else a revision such as `refs/pr/8`, or `main`).
PARENT_RUN_DIR = "" #@param {type:"string"}
#@markdown Optional: a finished SFT run folder (for example a previous notebook's output used as a Kaggle input).
UPLOAD = True #@param {type:"boolean"}
REPO_REF = "post-training-sft-rl" #@param {type:"string"}
#@markdown Git ref of centauri-bloom to run. Use `main` once the post-training PR is merged.

CHAT_REPO = "Centauri-Bloom/Bloom-Chat-0.2"
if ALGO not in ("grpo", "best-of-n"):
    raise ValueError("ALGO must be grpo or best-of-n.")
print("RL:", ALGO, "|", STEPS, "steps |", PROMPTS_PER_STEP, "prompts x", GROUP_SIZE, "samples | judge:", JUDGE_MODEL or "off")

In [ ]:
#@title 2. Set up the environment
import os, sys, subprocess, json, shutil
from pathlib import Path
from datetime import datetime, timezone

IN_KAGGLE = Path("/kaggle/working").is_dir()
os.environ["HF_HUB_DISABLE_XET"] = "1"

def checked(args):
    p = subprocess.run(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout, flush=True)
    if p.returncode:
        raise RuntimeError(f"Setup failed (exit {p.returncode}). See the full diagnostic above.")

def find_repo(start):
    cur = Path(start).resolve()
    for p in [cur, *cur.parents]:
        if (p / "pyproject.toml").is_file() and (p / "bloom" / "model.py").is_file():
            return p
    return None

if IN_KAGGLE:
    REPO = Path("/kaggle/working/centauri-bloom")
    REF = (REPO_REF or "main").strip() or "main"
    if not (REPO / ".git").exists():
        checked(["git", "clone", "--depth", "1", "--branch", REF, "https://github.com/InterstellarNerd8760/centauri-bloom.git", str(REPO)])
    else:
        checked(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", REF])
        checked(["git", "-C", str(REPO), "checkout", "-B", REF, "FETCH_HEAD"])
    BASE = Path("/kaggle/working/centauri-bloom-runs")
else:
    REPO = find_repo(Path.cwd())
    if REPO is None:
        raise RuntimeError("Run this on Kaggle, or from a centauri-bloom checkout.")
    BASE = REPO / "runs"
os.chdir(REPO)
BASE.mkdir(parents=True, exist_ok=True)
# Keep the preinstalled CUDA PyTorch; add only what the trainer needs.
checked([sys.executable, "-m", "pip", "install", "--no-deps", "-e", "."])
checked([sys.executable, "-m", "pip", "install", "tokenizers", "datasets", "huggingface_hub", "safetensors", "gguf"])

from bloom.notebook_workflow import submit_candidate, sha256, pick_device, preflight, upload_base_revision
DEVICE = pick_device()
if IN_KAGGLE and DEVICE != "cuda":
    raise RuntimeError("Enable a GPU accelerator (T4) in the Kaggle notebook settings, then restart.")
if DEVICE == "cuda" and shutil.which("nvidia-smi"):
    checked(["nvidia-smi"])
preflight(DEVICE)

In [ ]:
#@title 3. Authorize Hugging Face (candidate PR only)
from getpass import getpass
from bloom.notebook_workflow import auth_client
try:
    if IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    else:
        token = os.environ.get("HF_TOKEN")
except Exception:
    token = None
if not token:
    token = getpass("Hugging Face write token: ")
AUTH_API = auth_client(CHAT_REPO, token, require_repo=True)
del token
print("Authenticated. This notebook can only open a PR on", CHAT_REPO, "- it never writes to main and never creates repos.")

In [ ]:
#@title 4. Run RL (new run folder; the parent is never modified)
if JUDGE_MODEL.strip():
    checked([sys.executable, "-m", "pip", "install", "-q", "transformers"])
OUT_DIR = BASE / ("bloom-50m-rl-" + ALGO + "-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S"))
settings = dict(algo=ALGO, steps=STEPS, prompts_per_step=PROMPTS_PER_STEP, group_size=GROUP_SIZE, max_new=MAX_NEW,
                temperature=TEMPERATURE, lr=LEARNING_RATE, beta=KL_BETA, eval_every=EVAL_EVERY)
payload = json.dumps(dict(settings=settings, repo_dir=str(REPO), out_dir=str(OUT_DIR), device=DEVICE,
                          parent_run_dir=PARENT_RUN_DIR or None,
                          parent_hf_repo=None if PARENT_RUN_DIR else CHAT_REPO,
                          parent_hf_revision=None if PARENT_RUN_DIR else (PARENT_HF_REVISION.strip() or "auto"),
                          judge_model=JUDGE_MODEL.strip(), use_gsm8k=USE_GSM8K))
script = ("import sys; from bloom.rl_run import run_from_payload; run_from_payload(sys.argv[1])")
env = os.environ.copy()
env["HF_TOKEN"] = AUTH_API.token
try:
    result = subprocess.run([sys.executable, "-u", "-c", script, payload], env=env)
finally:
    env.pop("HF_TOKEN", None)
if result.returncode:
    raise RuntimeError("RL run failed; see the log above. Files (if any) are in " + str(OUT_DIR))
metrics = json.loads((OUT_DIR / "metrics.json").read_text())
print("Run folder:", OUT_DIR)

In [ ]:
#@title 5. Parent vs candidate (greedy, locked eval + held-out tasks)
ev = json.loads((OUT_DIR / "evaluation.json").read_text())
for who in ("parent", "candidate"):
    h, l = ev[who]["holdout"], ev[who]["locked"]
    print("%-9s composite %.4f | held-out accuracy %.3f | locked arithmetic %.3f | repetition %.3f | ended %.3f | tokens %.1f" % (
        who, ev[who]["composite"], h["verifiable_accuracy"] or 0, l["arithmetic_exact"] or 0, h["repetition_rate"], h["ended_rate"], h["mean_tokens"]))
print("Selected step:", ev["selected_step"], "| gain:", round(ev["composite_gain"], 4), "| uploadable:", metrics["candidate_uploadable"], "| stopped:", ev["stopped"])
print("By task type (candidate):", json.dumps(ev["candidate"]["holdout"]["by_type"], indent=1))
print(json.dumps(ev["candidate_samples"][:12], indent=1))

In [ ]:
#@title 6. Open the candidate PR (never main)
from bloom.posttrain import assert_upload_clean, provenance_is_private_safe
if not provenance_is_private_safe(json.loads((OUT_DIR / "data-provenance.json").read_text())):
    raise RuntimeError("data-provenance.json contains raw example fields; refusing to upload.")
if not metrics["candidate_uploadable"]:
    print("RL did not beat the parent by the required margin. Nothing was written to Hugging Face.")
elif not UPLOAD:
    print("UPLOAD is off; nothing was sent to Hugging Face.")
else:
    assert_upload_clean(["model.safetensors", "config.json", "tokenizer.json", "metrics.json",
                         "data-provenance.json", "evaluation.json", "bloom-f32.gguf"])
    submit_candidate(AUTH_API, CHAT_REPO, OUT_DIR, metrics["additional_steps"], upload_base_revision(metrics), metrics["run_name"])
    print("A PR was opened on", CHAT_REPO, "- review it and merge it yourself. Main was not touched.")